## Importar librerías y cargar el modelo y la base vectorial

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer

# Rutas
VECTOR_STORE = Path("/home/jupyteruser/work/vector_store")
METADATA_FOLDER = Path("/home/jupyteruser/work/corpus_upeu/metadatos")

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

client = chromadb.PersistentClient(path=str(VECTOR_STORE))
collection = client.get_collection("corpus_upeu")
print(f"Conexion exitosa. Documentos en coleccion: {collection.count()}")


/usr/local/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Conexion exitosa. Documentos en coleccion: 3886


## Definir el banco de preguntas de prueba

In [2]:
# Banco de preguntas con categoria esperada (Issue 5.2)
PREGUNTAS = [
    ("¿Cuáles son los derechos del estudiante?", "B"),
    ("¿Cómo puedo reservar mi matrícula?", "B"),
    ("¿Cuál es el procedimiento para cambiar de carrera?", "B"),
    ("¿Qué sanciones existen en la universidad?", "A"),
    ("¿Cómo solicito una beca?", "B"),
    ("¿Cuál es el procedimiento para presentar una queja?", "A"),
    ("¿Qué dice el estatuto sobre el gobierno universitario?", "A"),
    ("¿Cómo se realiza un proyecto de investigación?", "C"),
    ("¿Qué servicios ofrece la universidad a los egresados?", "D"),
    ("¿Cuál es la política ambiental de la UPeU?", "E"),
]
print(f"Banco de preguntas: {len(PREGUNTAS)} consultas")


Banco de preguntas: 10 consultas


## Función para evaluar una consulta

In [3]:
def evaluar_consulta(consulta, categoria_esperada=None, n_resultados=3,
                       umbral_excelente=0.30, umbral_aceptable=0.50):
    """
    Evalua una consulta contra el vector store usando el modelo multilingue.

    Issues corregidos:
    - Usa query_embeddings con paraphrase-multilingual-MiniLM-L12-v2
      (NO el modelo por defecto de ChromaDB que es ingles)
    - Umbral 0.40 (aceptable) en lugar de 0.80 (Issue 5.1)
    - 3 niveles de calidad (Issue 5.1)
    - Acepta categoria_esperada para auditoria (Issue 5.2)
    """
    # Generar embedding con NUESTRO modelo (multilingue, espanol)
    query_embedding = model.encode([consulta], normalize_embeddings=True)[0].tolist()

    resultados = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_resultados,
        include=["documents", "metadatas", "distances"]
    )

    metas = resultados['metadatas'][0] if resultados['metadatas'][0] else []
    dists = resultados['distances'][0] if resultados['distances'][0] else [1.0]

    top1_dist = dists[0]

    if top1_dist < umbral_excelente:
        calidad = "excelente"
        cubierta = True
    elif top1_dist < umbral_aceptable:
        calidad = "aceptable"
        cubierta = True
    else:
        calidad = "no_cubierta"
        cubierta = False

    categoria_match = None
    if categoria_esperada and metas:
        categoria_match = (metas[0].get('categoria') == categoria_esperada)

    return {
        "consulta": consulta,
        "cubierta": cubierta,
        "calidad": calidad,
        "top1_dist": round(top1_dist, 4),
        "top1_doc": metas[0].get('documento', '?') if metas else '?',
        "top1_categoria": metas[0].get('categoria', '?') if metas else '?',
        "top1_articulo": metas[0].get('articulo', '') if metas else '',
        "categoria_esperada": categoria_esperada,
        "categoria_match": categoria_match,
    }


## Evaluar todo el banco de preguntas

In [4]:
resultados = []

for pregunta, cat_esp in PREGUNTAS:
    r = evaluar_consulta(pregunta, categoria_esperada=cat_esp)
    resultados.append(r)
    emoji = "✓" if r['cubierta'] else "✗"
    cat_match = "✓" if r['categoria_match'] else ("✗" if r['categoria_match'] is False else "?")
    print(f"{emoji} [{r['calidad']:11}] d={r['top1_dist']:.3f} cat={r['top1_categoria']}{cat_match} {r['top1_articulo']:30} {pregunta[:50]}")

# Resumen
df_res = pd.DataFrame(resultados)
print(f"\nCubiertas (aceptable+excelente): {df_res['cubierta'].sum()}/{len(df_res)}")
print(f"Excelentes: {(df_res['calidad']=='excelente').sum()}/{len(df_res)}")
print(f"Categoría match: {df_res['categoria_match'].sum()}/{df_res['categoria_match'].notna().sum()}")


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


✓ [excelente  ] d=0.149 cat=A✗ CAPÍTULO II                    ¿Cuáles son los derechos del estudiante?
✗ [no_cubierta] d=0.560 cat=B✓ Artículo 33°                   ¿Cómo puedo reservar mi matrícula?
✓ [aceptable  ] d=0.474 cat=B✓ Capítulo VI                    ¿Cuál es el procedimiento para cambiar de carrera?
✓ [excelente  ] d=0.182 cat=B✗ Artículo 238º                  ¿Qué sanciones existen en la universidad?
✗ [no_cubierta] d=0.767 cat=A✗ Artículo 43°                   ¿Cómo solicito una beca?
✓ [aceptable  ] d=0.350 cat=E✗ Artículo 22°                   ¿Cuál es el procedimiento para presentar una queja
✓ [excelente  ] d=0.249 cat=A✓ Artículo 186°                  ¿Qué dice el estatuto sobre el gobierno universita
✓ [excelente  ] d=0.191 cat=C✓ Artículo 149°                  ¿Cómo se realiza un proyecto de investigación?
✓ [excelente  ] d=0.290 cat=A✗ Artículo 870°                  ¿Qué servicios ofrece la universidad a los egresad
✓ [aceptable  ] d=0.393 cat=A✗ Artículo 671°    

## Análisis detallado por consulta

In [5]:
# Convertir resultados a DataFrame
df_resultados = pd.DataFrame(resultados)

# Mostrar tabla de resultados
cols_mostrar = ['consulta', 'calidad', 'cubierta', 'top1_dist', 'top1_doc', 'top1_categoria', 'categoria_esperada', 'categoria_match']
print("Resultados detallados:")
display(df_resultados[cols_mostrar])

# Identificar consultas no cubiertas
no_cubiertas = df_resultados[df_resultados['cubierta'] == False]
if len(no_cubiertas) > 0:
    print(f"\nConsultas con baja cobertura ({len(no_cubiertas)}):")
    for i, row in no_cubiertas.iterrows():
        print(f"\n  {row['consulta']}")
        print(f"  Distancia: {row['top1_dist']}  |  Documento: {row['top1_doc']}")
        print(f"  Categoría: {row['top1_categoria']} (esperada: {row['categoria_esperada']})")
else:
    print("\n¡Todas las consultas están cubiertas!")


Resultados detallados:


,consulta,calidad,cubierta,top1_dist,top1_doc,top1_categoria,categoria_esperada,categoria_match
0,¿Cuáles son los derechos del estudiante?,excelente,True,0.1493,REGLAMENTO GENERAL UPeU 2023,A,B,False
1,¿Cómo puedo reservar mi matrícula?,no_cubierta,False,0.5597,REGLAMENTO ADMISION 2025.v7,B,B,True
2,¿Cuál es el procedimiento para cambiar de carr...,aceptable,True,0.4738,REGLAMENTO MOVILIDAD ACADEMICA ESTUDIANTIL Y D...,B,B,True
3,¿Qué sanciones existen en la universidad?,excelente,True,0.1821,REGLAMENTO DE ESTUDIOS V5_2025,B,A,False
4,¿Cómo solicito una beca?,no_cubierta,False,0.7674,REGLAMENTO GENERAL UPeU 2023,A,B,False
5,¿Cuál es el procedimiento para presentar una q...,aceptable,True,0.3499,Reglamento Prevención Acoso y Hostigamiento S...,E,A,False
6,¿Qué dice el estatuto sobre el gobierno univer...,excelente,True,0.2488,REGLAMENTO GENERAL UPeU 2023,A,A,True
7,¿Cómo se realiza un proyecto de investigación?,excelente,True,0.1913,REGLAMENTO INVESTIGACION UPeU 2025 V8,C,C,True
8,¿Qué servicios ofrece la universidad a los egr...,excelente,True,0.2904,REGLAMENTO GENERAL UPeU 2023,A,D,False
9,¿Cuál es la política ambiental de la UPeU?,aceptable,True,0.3932,REGLAMENTO GENERAL UPeU 2023,A,E,False



Consultas con baja cobertura (2):

  ¿Cómo puedo reservar mi matrícula?
  Distancia: 0.5597  |  Documento: REGLAMENTO ADMISION 2025.v7
  Categoría: B (esperada: B)

  ¿Cómo solicito una beca?
  Distancia: 0.7674  |  Documento: REGLAMENTO GENERAL UPeU 2023
  Categoría: A (esperada: B)


## Guardar informe de cobertura

In [6]:
# Guardar resultados completos
INFORME_COBERTURA = METADATA_FOLDER / "informe_cobertura.csv"
df_resultados.to_csv(INFORME_COBERTURA, index=False, encoding='utf-8')
print(f"Informe de cobertura guardado en {INFORME_COBERTURA}")

# Crear resumen estadistico
resumen = {
    "Total consultas": len(df_resultados),
    "Cubiertas (aceptable+)": int(df_resultados['cubierta'].sum()),
    "Excelentes": int((df_resultados['calidad']=='excelente').sum()),
    "Aceptables": int((df_resultados['calidad']=='aceptable').sum()),
    "No cubiertas": int((~df_resultados['cubierta']).sum()),
    "Tasa cobertura": round(df_resultados['cubierta'].mean() * 100, 1),
    "Distancia promedio": round(df_resultados['top1_dist'].mean(), 4),
    "Distancia minima": round(df_resultados['top1_dist'].min(), 4),
    "Distancia maxima": round(df_resultados['top1_dist'].max(), 4),
    "Categoria match": f"{df_resultados['categoria_match'].sum()}/{df_resultados['categoria_match'].notna().sum()}"
}

print("\nResumen estadistico:")
for k, v in resumen.items():
    print(f"  {k}: {v}")


Informe de cobertura guardado en /home/jupyteruser/work/corpus_upeu/metadatos/informe_cobertura.csv

Resumen estadistico:
  Total consultas: 10
  Cubiertas (aceptable+): 8
  Excelentes: 5
  Aceptables: 3
  No cubiertas: 2
  Tasa cobertura: 80.0
  Distancia promedio: 0.3606
  Distancia minima: 0.1493
  Distancia maxima: 0.7674
  Categoria match: 4/10
